In [ ]:
# 🔥 Clean uninstall (important)
!pip uninstall -y langchain langchain-core langchain-community langchain-groq

# 🔥 Install compatible versions
!pip install langchain==0.1.16 langchain-core langchain-community langchain-groq requests

In [ ]:
# Install
!pip install -q langchain-groq requests

import os
import requests
from langchain_groq import ChatGroq

# 🔑 USER INPUT
groq_api_key = input("Enter your GROQ API Key: ")
weather_api_key = input("Enter your OpenWeather API Key: ")
city = input("Enter your city: ")

os.environ["GROQ_API_KEY"] = groq_api_key

# 🌦️ Get weather ONCE (no agent)
def get_weather(city):
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={weather_api_key}&units=metric"
    res = requests.get(url)
    data = res.json()

    if res.status_code != 200:
        return None, data.get("message", "error")

    return {
        "temp": data["main"]["temp"],
        "feels": data["main"]["feels_like"],
        "humidity": data["main"]["humidity"],
        "desc": data["weather"][0]["description"]
    }, None

weather, error = get_weather(city)

if error:
    print(f"❌ Error: {error}")
else:
    # 🤖 LLM
    llm = ChatGroq(
        temperature=0,
        model="llama-3.1-8b-instant"
    )

    prompt = f"""
    You are a smart daily planner.

    City: {city}
    Weather:
    - Temperature: {weather['temp']}C
    - Feels like: {weather['feels']}C
    - Humidity: {weather['humidity']}%
    - Condition: {weather['desc']}

    Create a realistic plan:
    - Avoid outdoor activities if too hot (>30C)
    - Suggest indoor activities if humid
    - Give morning, afternoon, evening plan
    """

    response = llm.invoke(prompt)

    print("\n📝 FINAL PLAN:\n")
    print(response.content)